# Bullish/Bearish Engulfing Candle on SPY
## Strategy Brief
The Bullish and Bearish Engulfing patterns are powerful reversal signals in technical analysis. A Bullish Engulfing pattern suggests a potential upward reversal, while a Bearish Engulfing pattern indicates a potential downward reversal. This strategy involves identifying these patterns on the SPY ETF and taking long or short positions accordingly. The results of this strategy can vary, but it aims to capitalize on short-term price reversals.
## References
- https://www.quantum-algo.com/blog/guides/engulfing-candle-complete-guide/

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters and constants needed for the strategy. These include the stock ticker, date range, and any other relevant settings.

In [ ]:
TICKER = 'SPY'
START_DATE = '2010-01-01'
END_DATE = pd.to_datetime('today').strftime('%Y-%m-%d')

### PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance and compute the Bullish and Bearish Engulfing patterns. The results will be plotted along with the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute Bullish/Bearish Engulfing
engulfing = pd.DataFrame(index=data.index)
engulfing['Bullish'] = (data['Close'] > data['Open'].shift(1)) & (data['Open'] < data['Close'].shift(1)) & (data['Close'].shift(1) < data['Open'].shift(1))
engulfing['Bearish'] = (data['Close'] < data['Open'].shift(1)) & (data['Open'] > data['Close'].shift(1)) & (data['Close'].shift(1) > data['Open'].shift(1))

# Plot
data['Close'].plot(figsize=(14, 7), label='Close Price')
plt.scatter(data.index, data['Close'], c=np.where(engulfing['Bullish'], 'g', np.nan), label='Bullish Engulfing')
plt.scatter(data.index, data['Close'], c=np.where(engulfing['Bearish'], 'r', np.nan), label='Bearish Engulfing')
plt.legend()
plt.title('SPY Price with Bullish/Bearish Engulfing Patterns')
plt.show()

### PHASE 3 - Strategy Engineering
We will define the signal logic based on the Bullish and Bearish Engulfing patterns and create a positions series to represent our strategy's trades.

In [ ]:
signals = pd.Series(index=data.index)
signals[engulfing['Bullish']] = 1  # Long on Bullish Engulfing
signals[engulfing['Bearish']] = -1  # Short on Bearish Engulfing
positions = signals.ffill().fillna(0)  # Carry forward positions

### PHASE 4 - Coding & Backtesting
Using the positions series, we will compute the strategy's daily returns and plot the equity curve.

In [ ]:
returns = data['Close'].pct_change()
strategy_returns = positions.shift(1) * returns
cumulative_returns = (1 + strategy_returns).cumprod()

# Plot equity curve
cumulative_returns.plot(figsize=(14, 7), label='Strategy Equity Curve')
plt.title('Equity Curve of Bullish/Bearish Engulfing Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Max Drawdown. We will also compare the strategy's performance to a buy-and-hold approach.

In [ ]:
def calculate_cagr(returns):
    n = len(returns) / 252  # Assuming 252 trading days in a year
    return (returns[-1] ** (1/n)) - 1

def calculate_sharpe(returns, risk_free_rate=0.01):
    return (returns.mean() - risk_free_rate/252) / returns.std() * np.sqrt(252)

def calculate_sortino(returns, risk_free_rate=0.01):
    downside_std = returns[returns < 0].std()
    return (returns.mean() - risk_free_rate/252) / downside_std * np.sqrt(252)

def calculate_calmar(returns):
    max_dd = calculate_max_drawdown(returns)
    return calculate_cagr(returns) / max_dd

def calculate_max_drawdown(returns):
    cum_returns = (1 + returns).cumprod()
    peak = cum_returns.expanding(min_periods=1).max()
    drawdown = (cum_returns - peak) / peak
    return drawdown.min()

strategy_cagr = calculate_cagr(cumulative_returns)
strategy_sharpe = calculate_sharpe(strategy_returns)
strategy_sortino = calculate_sortino(strategy_returns)
strategy_calmar = calculate_calmar(strategy_returns)
strategy_max_dd = calculate_max_drawdown(strategy_returns)

buy_and_hold_returns = (1 + returns).cumprod()
buy_and_hold_cagr = calculate_cagr(buy_and_hold_returns)
buy_and_hold_sharpe = calculate_sharpe(returns)
buy_and_hold_sortino = calculate_sortino(returns)
buy_and_hold_calmar = calculate_calmar(returns)
buy_and_hold_max_dd = calculate_max_drawdown(returns)

performance_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_dd],
    'Buy & Hold': [buy_and_hold_cagr, buy_and_hold_sharpe, buy_and_hold_sortino, buy_and_hold_calmar, buy_and_hold_max_dd]
})
print(performance_table)

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's signal, and print the recommended position.

In [ ]:
def get_today_signal():
    recent_data = yf.download(TICKER, start=pd.to_datetime('today') - pd.Timedelta(days=60), end=pd.to_datetime('today'))
    recent_engulfing = pd.DataFrame(index=recent_data.index)
    recent_engulfing['Bullish'] = (recent_data['Close'] > recent_data['Open'].shift(1)) & (recent_data['Open'] < recent_data['Close'].shift(1)) & (recent_data['Close'].shift(1) < recent_data['Open'].shift(1))
    recent_engulfing['Bearish'] = (recent_data['Close'] < recent_data['Open'].shift(1)) & (recent_data['Open'] > recent_data['Close'].shift(1)) & (recent_data['Close'].shift(1) > recent_data['Open'].shift(1))
    if recent_engulfing['Bullish'].iloc[-1]:
        print('Bullish Engulfing detected. Consider going long.')
    elif recent_engulfing['Bearish'].iloc[-1]:
        print('Bearish Engulfing detected. Consider going short.')
    else:
        print('No Engulfing pattern detected today.')

get_today_signal()